# Module 03: Train vs Eval Mode

**ECBS5200 Pre-Work** — Practical Deep Learning Engineering

In this notebook, you'll see firsthand that a PyTorch model produces
**different outputs** depending on whether it's in train mode or eval mode.
No GPU needed — this runs on CPU in under a minute.

## Setup

We need `transformers` for the model and tokenizer, `datasets` for a real
complaint, and `torch` for the mode switching and gradient control.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
from datasets import load_dataset

## Load model and tokenizer

We load ModernBERT-base with a classification head configured for our
consumer complaint classification task (113 categories after label cleanup in Week 1).

**Important detail:** ModernBERT-base ships with `dropout=0.0` — dropout
is disabled by default. That means train mode and eval mode would produce
identical outputs out of the box, which defeats our demo. So we'll set
dropout to 0.1 (a common fine-tuning value) to see the effect.

This is actually a useful lesson: **not all models have the same defaults.**
You should always check your model's configuration.

In [ ]:
model_name = "answerdotai/ModernBERT-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load config and set dropout to 0.1 so we can demonstrate the effect
config = AutoConfig.from_pretrained(model_name)
print(f"Default dropout in ModernBERT: {config.classifier_dropout}")

config.classifier_dropout = 0.1
config.hidden_dropout_prob = 0.1
config.num_labels = 111

model = AutoModelForSequenceClassification.from_pretrained(
    model_name, config=config
)

print(f"\nModel: {model_name}")
print(f"Dropout set to: 0.1 (for this demo)")
print(f"Current mode: {'train' if model.training else 'eval'}")

Notice the model loaded in **eval mode**. HuggingFace's `from_pretrained`
returns models in eval mode by default. In practice, you'll call
`model.train()` explicitly when you start training — never assume.

## Grab a real complaint

Let's take one real consumer complaint from our dataset and tokenize it.

In [ ]:
dataset = load_dataset("determined-ai/consumer_complaints_medium", split="train")

# Pick a complaint
complaint_text = dataset[0]["Consumer Complaint"]
print(f"Complaint: {complaint_text[:200]}...")

# Tokenize it
inputs = tokenizer(
    complaint_text,
    max_length=128,
    truncation=True,
    padding="max_length",
    return_tensors="pt",  # return PyTorch tensors
)

print(f"\nTokenized shape: {inputs['input_ids'].shape}")
print(f"Real tokens: {inputs['attention_mask'].sum().item()}")

## Experiment 1: Train mode — outputs differ each time

With the model in train mode, dropout is active. Let's pass the **exact same
input** through the model 5 times and see what comes out.

In [ ]:
model.train()
print(f"Mode: {'train' if model.training else 'eval'}")
print()

train_logits = []
for i in range(5):
    with torch.no_grad():  # we don't need gradients for this demo
        outputs = model(**inputs)
    logits = outputs.logits[0]  # shape: (111,)
    train_logits.append(logits)

    # Show the predicted class and the top logit value
    predicted_class = logits.argmax().item()
    top_logit = logits.max().item()
    print(f"  Run {i+1}: predicted class = {predicted_class:3d}, top logit = {top_logit:.4f}")

**Look at the results above.** The predicted class and/or the logit values
change between runs — even though the input is identical every time. This is
dropout at work: different neurons are randomly zeroed out on each forward pass.

In [ ]:
# Let's quantify how much the outputs differ
diffs = []
for i in range(1, len(train_logits)):
    diff = (train_logits[0] - train_logits[i]).abs().mean().item()
    diffs.append(diff)
    print(f"  Mean absolute difference (run 1 vs run {i+1}): {diff:.6f}")

print(f"\n  Average difference across runs: {sum(diffs)/len(diffs):.6f}")
print("  These are NOT zero — train mode introduces randomness via dropout.")

## Experiment 2: Eval mode — outputs are identical

Now let's switch to eval mode and repeat the same experiment.

In [ ]:
model.eval()
print(f"Mode: {'train' if model.training else 'eval'}")
print()

eval_logits = []
for i in range(5):
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits[0]
    eval_logits.append(logits)

    predicted_class = logits.argmax().item()
    top_logit = logits.max().item()
    print(f"  Run {i+1}: predicted class = {predicted_class:3d}, top logit = {top_logit:.4f}")

**Every run produces the exact same output.** Same predicted class, same logit
values, down to the last decimal place. That's because dropout is off — all
neurons are active, no randomness.

In [ ]:
# Verify they're truly identical
diffs_eval = []
for i in range(1, len(eval_logits)):
    diff = (eval_logits[0] - eval_logits[i]).abs().mean().item()
    diffs_eval.append(diff)
    print(f"  Mean absolute difference (run 1 vs run {i+1}): {diff:.6f}")

print(f"\n  Average difference across runs: {sum(diffs_eval)/len(diffs_eval):.6f}")
print("  These ARE zero — eval mode is deterministic.")

## Why this matters even when the default dropout is 0.0

You might wonder: if ModernBERT defaults to 0.0 dropout, why bother?

1. **You may add dropout during fine-tuning.** It's common to set dropout
   to 0.1 or 0.2 as regularization — and in Week 2, you might do exactly that.
2. **Other models use dropout by default.** BERT, DeBERTa, GPT-2 — most
   pretrained models ship with non-zero dropout. The habit must be automatic.
3. **`model.eval()` isn't just about dropout.** It also freezes batchnorm
   statistics and may affect other layer behaviors in future architectures.
4. **It's a professional habit.** Always calling `model.eval()` before
   inference is like always wearing a seatbelt — you don't skip it just
   because you're driving slowly today.

**The rule: always call `model.eval()` for inference. No exceptions.**

## Experiment 3: `torch.no_grad()` — memory savings

`torch.no_grad()` is separate from `model.eval()`. It tells PyTorch to skip
gradient computation, which saves memory. Let's see the difference.

In [ ]:
model.eval()

# --- WITH gradient tracking ---
outputs_with_grad = model(**inputs)
logits_with_grad = outputs_with_grad.logits[0]

# --- WITHOUT gradient tracking ---
with torch.no_grad():
    outputs_no_grad = model(**inputs)
    logits_no_grad = outputs_no_grad.logits[0]

# Verify the outputs are identical
diff = (logits_with_grad - logits_no_grad).abs().max().item()
print(f"Max difference in outputs: {diff:.10f}")
print(f"Outputs are identical: {diff == 0.0}")
print()
print("torch.no_grad() does NOT change the result.")
print("It only skips gradient computation to save memory.")
print()
print("On a GPU with large batches, this can mean the difference")
print("between fitting in memory and getting an OOM (out-of-memory) error.")

Key insight: `torch.no_grad()` produces **exactly the same output** as without it.
It just uses less memory by not tracking the computation graph needed for
backpropagation.

## The complete pattern

Here's what you'll write every week in this course:

In [ ]:
# === THE PATTERN ===
# This is pseudocode showing the structure — don't run this as-is.

print("""
# TRAINING LOOP
model.train()                          # dropout ON, batchnorm updating
for batch in train_loader:
    outputs = model(**batch)
    loss = outputs.loss
    loss.backward()                    # compute gradients
    optimizer.step()                   # update weights
    optimizer.zero_grad()              # reset gradients

# EVALUATION LOOP
model.eval()                           # dropout OFF, batchnorm frozen
with torch.no_grad():                  # don't track gradients (save memory)
    for batch in eval_loader:
        outputs = model(**batch)
        predictions = outputs.logits.argmax(dim=-1)
        # compute metrics — accuracy, F1, etc.
""")

## Summary

| | `model.train()` | `model.eval()` |
|---|---|---|
| **Dropout** | ON (random) | OFF (deterministic) |
| **BatchNorm** | Updates running stats | Uses frozen stats |
| **When to use** | Training loop | Validation / test / deployment |

| | With gradients | `torch.no_grad()` |
|---|---|---|
| **Output** | Same | Same |
| **Memory** | Higher (stores computation graph) | Lower (no graph) |
| **When to use** | Training (need `loss.backward()`) | Inference (no backprop needed) |

## Check yourself

Before moving to Module 04, make sure you can answer these:

1. Why does the same input produce different outputs in train mode
   (when dropout > 0)?
2. What does `model.eval()` actually change inside the model?
3. Does `torch.no_grad()` change the model's output? What does it do?
4. What happens to your validation metrics if you forget to call
   `model.eval()` and your model uses dropout?
5. Why should you always call `model.eval()` for inference, even if your
   model's default dropout is 0.0?